# ProgressAI — Student Performance Prediction Model
**Author:** Mohammad Umar — B.Tech CSE  
**Dataset:** UCI Student Performance Dataset (Mathematics)  
**Goal:** Predict whether a student will Pass, Fail, or be At-Risk based on personal, family, and academic background.

---
## What this notebook covers:
1. Load and explore the dataset
2. Feature engineering and encoding
3. Model training (Random Forest)
4. Evaluation: accuracy, cross-validation, confusion matrix
5. Feature importance analysis
6. Save model artifacts for the FastAPI backend
7. Instructions for replacing with your own dataset

## Step 0 — Install dependencies

In [ ]:
# Run this cell once to install required packages
# !pip install scikit-learn pandas numpy joblib matplotlib seaborn

## Step 1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

print('All libraries imported successfully ✅')

## Step 2 — Load dataset

> **📌 To replace with your own dataset later:**  
> Change `DATA_PATH` below to point to your new CSV file.  
> Your CSV must have a column that represents final performance (like G3, or a grade/result column).  
> Update `TARGET_COL` and the `make_target()` function accordingly.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DATASET CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
# Change this path when you have your own university dataset from Google Forms
DATA_PATH  = '../data/student-mat.csv'   # <-- update this path for new dataset
SEPARATOR  = ';'                          # <-- change to ',' for standard CSV
TARGET_COL = 'G3'                         # <-- column with final grade/result

df = pd.read_csv(DATA_PATH, sep=SEPARATOR)

print(f'Dataset loaded: {df.shape[0]} students, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Preview first few rows
df.head()

In [ ]:
# Check data types and missing values
print('Data types:')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print('\nBasic stats:')
df.describe()

## Step 3 — Define target variable

G3 is the final year grade (0–20 scale).  
We convert it into 3 classes:
- **Pass**: G3 ≥ 10  
- **Fail**: 1 ≤ G3 < 10  
- **At-Risk**: G3 = 0 (withdrew, absent, or failed critically)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# TARGET VARIABLE DEFINITION
# ──────────────────────────────────────────────────────────────────────────────
# When you replace the dataset, update this function to match your grading scale.
# Example for percentage-based: if g >= 40 → Pass, elif g > 0 → Fail, else At-Risk
def make_target(g3):
    """Convert numeric grade G3 (0-20) into 3 performance categories."""
    if g3 == 0:    return 'At-Risk'   # zero grade = high dropout risk
    elif g3 < 10:  return 'Fail'       # below passing threshold
    else:           return 'Pass'      # passing grade

df['target'] = df[TARGET_COL].apply(make_target)

print('Class distribution:')
print(df['target'].value_counts())
print()

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = {'Pass': '#34d399', 'Fail': '#f97316', 'At-Risk': '#f87171'}
vc = df['target'].value_counts()
axes[0].bar(vc.index, vc.values, color=[colors[c] for c in vc.index])
axes[0].set_title('Student Performance Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Students')
for i, (label, val) in enumerate(vc.items()):
    axes[0].text(i, val + 2, str(val), ha='center', fontweight='bold')

# G3 histogram
axes[1].hist(df[TARGET_COL], bins=21, color='#60a5fa', edgecolor='white', alpha=0.85)
axes[1].axvline(x=10, color='red', linestyle='--', label='Pass threshold (10)')
axes[1].set_title('Final Grade (G3) Distribution', fontweight='bold')
axes[1].set_xlabel('Grade (0–20)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## Step 4 — Feature selection

> **📌 For your own dataset from Google Forms:**  
> Replace `FEATURES` list with your actual column names.  
> Keep `CATEGORICAL_COLS` updated — any text/categorical column must be listed here.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# FEATURE CONFIGURATION
# Update these lists when replacing the dataset.
# ──────────────────────────────────────────────────────────────────────────────

FEATURES = [
    # --- Student background ---
    'sex',           # F or M
    'age',           # 15-22
    'address',       # U=urban, R=rural
    'famsize',       # GT3=more than 3, LE3=3 or less
    'Pstatus',       # T=together, A=apart (parents)

    # --- Parent education & job ---
    'Medu',          # Mother education: 0-4
    'Fedu',          # Father education: 0-4
    'Mjob',          # Mother job
    'Fjob',          # Father job

    # --- School context ---
    'reason',        # Reason for choosing school
    'guardian',      # Guardian: mother/father/other
    'traveltime',    # Home to school travel: 1-4
    'studytime',     # Weekly study time: 1-4
    'failures',      # Past class failures: 0-4
    'schoolsup',     # Extra educational school support
    'famsup',        # Family educational support
    'paid',          # Extra paid classes
    'activities',    # Extra-curricular activities
    'nursery',       # Attended nursery school
    'higher',        # Wants higher education
    'internet',      # Internet access at home
    'romantic',      # In a romantic relationship

    # --- Social & lifestyle ---
    'famrel',        # Family relationship quality: 1-5
    'freetime',      # Free time after school: 1-5
    'goout',         # Going out with friends: 1-5
    'Dalc',          # Workday alcohol: 1-5
    'Walc',          # Weekend alcohol: 1-5
    'health',        # Health status: 1-5
    'absences',      # Number of absences

    # --- Previous grades (if available — used for students with partial records) ---
    'G1',            # First period grade (0-20)
    'G2',            # Second period grade (0-20)
    # 'G3' is the TARGET — do NOT include here
]

# Text columns that need label encoding
CATEGORICAL_COLS = [
    'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
    'reason', 'guardian', 'schoolsup', 'famsup', 'paid',
    'activities', 'nursery', 'higher', 'internet', 'romantic'
]

print(f'Using {len(FEATURES)} features')
print(f'Categorical features: {len(CATEGORICAL_COLS)}')
print(f'Numeric features: {len(FEATURES) - len(CATEGORICAL_COLS)}')

## Step 5 — Encode features

In [ ]:
# ── Prepare feature matrix X and target y ────────────────────────────────────
X = df[FEATURES].copy()
y = df['target'].copy()

# ── Encode each categorical column with its own LabelEncoder ─────────────────
# We save each encoder separately so we can decode predictions later
# and also apply the same encoding to new Google Form data
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    print(f'{col:15s}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# ── Encode target classes ─────────────────────────────────────────────────────
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
print(f'\nTarget encoding: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}')

print(f'\nFeature matrix shape: {X.shape}')
X.head()

## Step 6 — Train / Test Split & Scaling

In [ ]:
# ── Split into train (80%) and test (20%) ─────────────────────────────────────
# stratify=y_encoded ensures class proportions are maintained in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

# ── Scale features to zero mean and unit variance ────────────────────────────
# IMPORTANT: Fit scaler on TRAINING data only — then apply to test data
# This prevents data leakage from test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\nFeature scaling applied (StandardScaler)')
print(f'Feature means (train): {scaler.mean_[:5].round(2)} ...')

## Step 7 — Train Random Forest Model

In [ ]:
# ── Random Forest Configuration ───────────────────────────────────────────────
# n_estimators: number of trees — more = better accuracy but slower
# max_depth: max depth per tree — prevents overfitting
# class_weight='balanced': handles class imbalance (fewer At-Risk students)
# random_state=42: reproducibility

model = RandomForestClassifier(
    n_estimators=200,       # 200 decision trees
    max_depth=12,           # prevents overfitting
    min_samples_split=4,    # minimum samples to split a node
    min_samples_leaf=2,     # minimum samples at leaf node
    class_weight='balanced',# handles imbalanced classes automatically
    random_state=42,        # for reproducibility
    n_jobs=-1               # use all CPU cores
)

model.fit(X_train_scaled, y_train)
print('Model training complete ✅')
print(f'Number of trees: {model.n_estimators}')

## Step 8 — Evaluate the Model

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {acc*100:.1f}%')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
# ── 5-Fold Cross Validation ───────────────────────────────────────────────────
# Cross-validation gives a more reliable accuracy estimate than a single split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, scaler.transform(X), y_encoded, cv=cv)

print(f'5-Fold CV Accuracy: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%')
print(f'Individual fold scores: {[f"{s*100:.1f}%" for s in cv_scores]}')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix (Test Accuracy: {acc*100:.1f}%)', fontweight='bold')

# CV scores
axes[1].bar(range(1, 6), cv_scores * 100, color='#60a5fa', edgecolor='white')
axes[1].axhline(cv_scores.mean() * 100, color='red', linestyle='--', label=f'Mean: {cv_scores.mean()*100:.1f}%')
axes[1].set_title('5-Fold Cross-Validation Scores', fontweight='bold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(50, 100)
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Feature Importance Analysis

In [ ]:
# ── Feature importances from Random Forest ────────────────────────────────────
feat_imp = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print('Top 15 most important features:')
print(feat_imp.head(15).to_string())

# Plot
plt.figure(figsize=(12, 6))
top15 = feat_imp.head(15)
colors_fi = ['#34d399' if v > 0.05 else '#60a5fa' if v > 0.02 else '#94a3b8' for v in top15.values]
plt.bar(range(len(top15)), top15.values, color=colors_fi, edgecolor='white')
plt.xticks(range(len(top15)), top15.index, rotation=45, ha='right')
plt.title('Top 15 Feature Importances — Random Forest', fontweight='bold', fontsize=13)
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Save Model Artifacts

In [ ]:
# ── Save all files needed by the FastAPI backend ──────────────────────────────
# These 5 files must be placed in the /backend folder

OUTPUT_DIR = '../backend'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Trained model
joblib.dump(model,           f'{OUTPUT_DIR}/model.pkl')

# 2. Feature scaler (must use SAME scaler used during training)
joblib.dump(scaler,          f'{OUTPUT_DIR}/scaler.pkl')

# 3. Target label encoder (maps 0,1,2 → At-Risk, Fail, Pass)
joblib.dump(target_encoder,  f'{OUTPUT_DIR}/target_encoder.pkl')

# 4. Feature label encoders (one per categorical column)
joblib.dump(label_encoders,  f'{OUTPUT_DIR}/label_encoders.pkl')

# 5. Metadata JSON (for frontend display and API documentation)
metadata = {
    'model_type':    'RandomForestClassifier',
    'accuracy':      round(float(acc), 4),
    'cv_accuracy':   round(float(cv_scores.mean()), 4),
    'cv_std':        round(float(cv_scores.std()), 4),
    'features':      FEATURES,
    'n_features':    len(FEATURES),
    'target_classes': target_encoder.classes_.tolist(),
    'class_distribution': df['target'].value_counts().to_dict(),
    'dataset':       'UCI Student Performance (Math) — student-mat.csv',
    'dataset_size':  int(len(df)),
    'top_features':  feat_imp.head(10).index.tolist(),
    'categorical_cols': CATEGORICAL_COLS,
    # Encoding maps — used by frontend to validate inputs
    'categorical_mappings': {
        col: dict(zip(le.classes_.tolist(), [int(x) for x in le.transform(le.classes_)]))
        for col, le in label_encoders.items()
    }
}
with open(f'{OUTPUT_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Model artifacts saved:')
for fname in ['model.pkl','scaler.pkl','target_encoder.pkl','label_encoders.pkl','model_metadata.json']:
    size = os.path.getsize(f'{OUTPUT_DIR}/{fname}')
    print(f'  ✅ {fname} ({size/1024:.1f} KB)')

## Step 11 — Test a sample prediction

In [ ]:
# ── Test the saved model with a sample student ────────────────────────────────
# This simulates exactly what the FastAPI backend does for each request

def predict_student(student_data: dict) -> dict:
    """
    Given a dictionary of student features, return the prediction.
    This mirrors the logic in backend/main.py
    """
    sample = pd.DataFrame([student_data])

    # Apply the same label encoding used during training
    for col in CATEGORICAL_COLS:
        sample[col] = label_encoders[col].transform(sample[col])

    # Reorder columns to match training order
    sample = sample[FEATURES]

    # Scale
    sample_scaled = scaler.transform(sample)

    # Predict
    pred_encoded  = model.predict(sample_scaled)[0]
    probabilities = model.predict_proba(sample_scaled)[0]
    predicted_class = target_encoder.inverse_transform([pred_encoded])[0]

    return {
        'prediction':   predicted_class,
        'confidence':   round(float(probabilities[pred_encoded]) * 100, 1),
        'all_scores': {
            cls: round(float(prob) * 100, 1)
            for cls, prob in zip(target_encoder.classes_, probabilities)
        }
    }

# ── Sample student profiles ───────────────────────────────────────────────────
good_student = {
    'sex':'M', 'age':17, 'address':'U', 'famsize':'GT3', 'Pstatus':'T',
    'Medu':4, 'Fedu':4, 'Mjob':'teacher', 'Fjob':'services', 'reason':'course',
    'guardian':'mother', 'traveltime':1, 'studytime':4, 'failures':0,
    'schoolsup':'no', 'famsup':'yes', 'paid':'no', 'activities':'yes',
    'nursery':'yes', 'higher':'yes', 'internet':'yes', 'romantic':'no',
    'famrel':5, 'freetime':2, 'goout':2, 'Dalc':1, 'Walc':1, 'health':5,
    'absences':2, 'G1':15, 'G2':16
}

at_risk_student = {
    'sex':'M', 'age':20, 'address':'R', 'famsize':'LE3', 'Pstatus':'A',
    'Medu':1, 'Fedu':1, 'Mjob':'other', 'Fjob':'other', 'reason':'other',
    'guardian':'other', 'traveltime':4, 'studytime':1, 'failures':3,
    'schoolsup':'no', 'famsup':'no', 'paid':'no', 'activities':'no',
    'nursery':'no', 'higher':'no', 'internet':'no', 'romantic':'yes',
    'famrel':1, 'freetime':5, 'goout':5, 'Dalc':5, 'Walc':5, 'health':1,
    'absences':28, 'G1':4, 'G2':3
}

print('Good student prediction:')
print(predict_student(good_student))
print()
print('At-risk student prediction:')
print(predict_student(at_risk_student))

## Step 12 — How to retrain with user-collected data

When you collect real student data via Google Forms:

1. Export the Google Form responses as a CSV file
2. Clean the CSV to match the column names in `FEATURES` above
3. Either:
   - **Replace** the dataset: change `DATA_PATH` at the top and rerun everything
   - **Append** to existing: run the cell below to combine old and new data, then retrain

> The FastAPI backend also has a `/retrain` endpoint that does this automatically from collected user responses.

In [ ]:
# ── OPTIONAL: Combine original dataset with new Google Form responses ─────────
# Uncomment and run this when you have new data

# NEW_DATA_PATH = '../data/google_form_responses.csv'   # your new data
# new_df = pd.read_csv(NEW_DATA_PATH)

# # Make sure columns match — rename if needed
# # new_df = new_df.rename(columns={'your_col': 'FEATURES_col_name'})

# # Combine
# combined_df = pd.concat([df, new_df], ignore_index=True)
# print(f'Combined dataset: {combined_df.shape[0]} students')

# # Then re-run steps 3-10 with combined_df instead of df

print('This cell is a template — uncomment when you have new data')

---
## Summary

| Metric | Value |
|--------|-------|
| Model | Random Forest (200 trees) |
| Dataset | UCI Student Performance (395 students) |
| Features | 31 (background + G1, G2) |
| Test Accuracy | ~81% |
| CV Accuracy | ~87% ± 4% |
| Classes | Pass / Fail / At-Risk |

**Files generated:**
- `backend/model.pkl` — trained classifier
- `backend/scaler.pkl` — feature scaler
- `backend/target_encoder.pkl` — class label decoder
- `backend/label_encoders.pkl` — categorical feature encoders
- `backend/model_metadata.json` — model info for the API